In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("telecom.bronze.sip_packets")

# 1. Fix data types
typed = (bronze
    .withColumn("frame_number",   F.col("frame_number").cast("int"))
    .withColumn("event_time_utc", F.col("time_epoch").cast("double").cast("timestamp"))
    .withColumn("status_code",    F.col("status_code").cast("int"))
    .withColumn("cseq_num",       F.split("cseq", " ")[0].cast("long"))
    .withColumn("cseq_method",    F.split("cseq", " ")[1])
)

# 2. Deduplicate: same message captured twice -> keep the earliest
w = Window.partitionBy("call_id", "cseq", "method", "status_code").orderBy("event_time_utc")

silver = (typed
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("event_time_utc", "frame_number", "src_ip", "dst_ip",
            "call_id", "cseq_num", "cseq_method", "method", "status_code",
            "source_file", "ingested_at")
)

# 3. Save as a Delta table
silver.write.mode("overwrite").saveAsTable("telecom.silver.sip_messages")

display(spark.table("telecom.silver.sip_messages"))